# Limpieza y Preprocesamiento de PDFs para RAG

## Introducción

Antes de alimentar documentos a un sistema RAG, es fundamental **limpiar y preprocesar** el texto extraído de los PDFs. Los PDFs suelen contener artefactos como encabezados, pies de página, números de página, saltos de línea innecesarios, caracteres especiales y texto mal codificado que degradan la calidad de las búsquedas semánticas y las respuestas del LLM.

Este notebook cubre:

1. **Extracción de texto** de PDFs usando múltiples librerías
2. **Limpieza de texto** — eliminar artefactos, normalizar espacios, corregir codificación
3. **Segmentación inteligente** (chunking) — dividir documentos largos en fragmentos óptimos
4. **Exportación** — generar objetos `Document` de LangChain listos para el pipeline RAG


## 1. Instalación de dependencias

In [ ]:
# Instalar librerías para extracción y limpieza de PDFs
%pip install -q PyPDF2 pdfplumber pymupdf
%pip install -q langchain langchain-core langchain-text-splitters
%pip install -q unidecode

print("Dependencias instaladas correctamente")

## 2. Importar librerías

In [ ]:
import os
import re
import glob
from pathlib import Path
from typing import Optional

# Extracción de PDFs
import PyPDF2
import pdfplumber
import fitz  # PyMuPDF

# LangChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Librerías importadas correctamente")

## 3. Configuración

Define la carpeta donde están tus PDFs. Puedes cambiar la ruta según tu estructura de archivos.

In [ ]:
# Carpeta con los PDFs a procesar
PDF_FOLDER = "./pdfs"  # Cambia esta ruta a tu carpeta de PDFs

# Crear la carpeta si no existe
os.makedirs(PDF_FOLDER, exist_ok=True)

# Listar PDFs disponibles
pdf_files = sorted(glob.glob(os.path.join(PDF_FOLDER, "*.pdf")))

if pdf_files:
    print(f"Se encontraron {len(pdf_files)} archivos PDF:")
    for f in pdf_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  - {os.path.basename(f)} ({size_mb:.2f} MB)")
else:
    print(f"No se encontraron PDFs en '{PDF_FOLDER}'.")
    print(f"Coloca tus archivos PDF en la carpeta '{PDF_FOLDER}' y vuelve a ejecutar esta celda.")

## 4. Extracción de texto desde PDFs

Ofrecemos tres métodos de extracción. Cada librería tiene fortalezas distintas:

| Librería | Fortaleza | Limitación |
|----------|-----------|------------|
| **PyPDF2** | Rápida, ligera | Pierde formato en PDFs complejos |
| **pdfplumber** | Buena con tablas y layouts | Más lenta |
| **PyMuPDF (fitz)** | Rápida, precisa, maneja imágenes | Más pesada |

Usamos PyMuPDF como método principal por su balance entre velocidad y calidad.

In [ ]:
def extract_text_pymupdf(pdf_path: str) -> list[dict]:
    """Extrae texto página por página usando PyMuPDF (recomendado)."""
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text")
            if text.strip():
                pages.append({
                    "text": text,
                    "page": page_num,
                    "total_pages": len(doc),
                    "source": os.path.basename(pdf_path)
                })
    return pages


def extract_text_pypdf2(pdf_path: str) -> list[dict]:
    """Extrae texto página por página usando PyPDF2 (alternativa ligera)."""
    pages = []
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({
                    "text": text,
                    "page": page_num,
                    "total_pages": len(reader.pages),
                    "source": os.path.basename(pdf_path)
                })
    return pages


def extract_text_pdfplumber(pdf_path: str) -> list[dict]:
    """Extrae texto página por página usando pdfplumber (bueno para tablas)."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({
                    "text": text,
                    "page": page_num,
                    "total_pages": len(pdf.pages),
                    "source": os.path.basename(pdf_path)
                })
    return pages


# Diccionario de métodos disponibles
EXTRACTORS = {
    "pymupdf": extract_text_pymupdf,
    "pypdf2": extract_text_pypdf2,
    "pdfplumber": extract_text_pdfplumber,
}

print("Funciones de extracción definidas:")
for name in EXTRACTORS:
    print(f"  - {name}")

## 5. Funciones de limpieza de texto

Aquí definimos las funciones que eliminan artefactos comunes en PDFs:

- **Encabezados y pies de página** repetitivos
- **Números de página** sueltos
- **Saltos de línea** dentro de párrafos (hyphenation)
- **Espacios múltiples** y líneas en blanco
- **Caracteres especiales** y basura Unicode
- **Referencias y notas al pie** (opcional)

In [ ]:
def remove_page_numbers(text: str) -> str:
    """Elimina números de página sueltos (líneas que solo contienen un número)."""
    lines = text.split("\n")
    cleaned = [line for line in lines if not re.match(r"^\s*\d{1,4}\s*$", line)]
    return "\n".join(cleaned)


def remove_headers_footers(text: str, patterns: Optional[list[str]] = None) -> str:
    """Elimina encabezados y pies de página basándose en patrones."""
    if patterns is None:
        # Patrones comunes en documentos académicos/técnicos
        patterns = [
            r"^\s*Page\s+\d+\s*(of\s+\d+)?\s*$",       # "Page 5 of 10"
            r"^\s*Página\s+\d+\s*(de\s+\d+)?\s*$",     # "Página 5 de 10"
            r"^\s*-\s*\d+\s*-\s*$",                     # "- 5 -"
            r"^\s*\d+\s*/\s*\d+\s*$",                   # "5/10"
            r"^\s*©.*$",                                 # Líneas de copyright
            r"^\s*All rights reserved\.?\s*$",           # Derechos reservados
            r"^\s*Todos los derechos reservados\.?\s*$",
            r"^\s*Confidential\.?\s*$",                  # Confidencialidad
            
        ]

    lines = text.split("\n")
    cleaned = []
    for line in lines:
        if not any(re.match(p, line, re.IGNORECASE) for p in patterns):
            cleaned.append(line)
    return "\n".join(cleaned)


def fix_hyphenation(text: str) -> str:
    """Reconecta palabras cortadas por guión al final de línea."""
    # "progra-\nming" -> "programming"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)
    return text


def normalize_whitespace(text: str) -> str:
    """Normaliza espacios en blanco: colapsa múltiples espacios y líneas vacías."""
    # Reemplazar múltiples espacios por uno solo
    text = re.sub(r"[^\S\n]+", " ", text)
    # Reemplazar 3+ líneas vacías por 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Limpiar espacios al inicio/final de cada línea
    lines = [line.strip() for line in text.split("\n")]
    return "\n".join(lines)


def remove_special_characters(text: str) -> str:
    """Elimina caracteres de control y basura Unicode, preservando acentos y ñ."""
    # Eliminar caracteres de control (excepto newlines y tabs)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    # Reemplazar caracteres Unicode problemáticos comunes
    replacements = {
        "\u2018": "'", "\u2019": "'",  # Comillas simples tipográficas
        "\u201c": '"', "\u201d": '"',  # Comillas dobles tipográficas
        "\u2013": "-", "\u2014": "-",  # Guiones em/en
        "\u2026": "...",                 # Elipsis
        "\u00a0": " ",                   # Non-breaking space
        "\ufeff": "",                     # BOM
        "\u200b": "",                     # Zero-width space
        "\uf0b7": "- ",                   # Bullet point (symbol font)
        "\uf0a7": "- ",                   # Otro bullet
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def join_broken_paragraphs(text: str) -> str:
    """Une líneas que son continuación de un párrafo (no terminan en punto, etc.)."""
    lines = text.split("\n")
    result = []
    for i, line in enumerate(lines):
        if not line.strip():
            result.append(line)
            continue

        # Si la línea anterior no termina en puntuación final y la actual
        # empieza en minúscula, probablemente es continuación del párrafo
        if (result
            and result[-1].strip()
            and not re.search(r"[.!?:;]\s*$", result[-1])
            and re.match(r"^[a-záéíóúüñ]", line.strip())):
            result[-1] = result[-1].rstrip() + " " + line.strip()
        else:
            result.append(line)
    return "\n".join(result)


print("Funciones de limpieza definidas correctamente")

## 6. Pipeline de limpieza completo

Combinamos todas las funciones de limpieza en un pipeline configurable.

In [ ]:
def clean_text(
    text: str,
    remove_pages: bool = True,
    remove_hf: bool = True,
    fix_hyphens: bool = True,
    join_paragraphs: bool = True,
    clean_special: bool = True,
    normalize_ws: bool = True,
    custom_header_patterns: Optional[list[str]] = None,
) -> str:
    """Pipeline de limpieza completo. Cada paso se puede activar/desactivar."""

    if clean_special:
        text = remove_special_characters(text)

    if remove_pages:
        text = remove_page_numbers(text)

    if remove_hf:
        text = remove_headers_footers(text, patterns=custom_header_patterns)

    if fix_hyphens:
        text = fix_hyphenation(text)

    if join_paragraphs:
        text = join_broken_paragraphs(text)

    if normalize_ws:
        text = normalize_whitespace(text)

    return text.strip()


# Demostración con texto de ejemplo
sample_dirty_text = """   Page 1 of 5

   Introduction to Machine   Learning

Machine learning is a sub\u00adfield of artifi-
cial intelligence that focuses on the deve-
lopment of algorithms that can learn from
and make predictions on data.




\u201cDeep learning\u201d is a subset of machine lear-
ning based on artificial neural networks.

   2

© 2024 All rights reserved.
"""

print("=== TEXTO ORIGINAL ===")
print(repr(sample_dirty_text[:200]))
print()

cleaned = clean_text(sample_dirty_text)
print("=== TEXTO LIMPIO ===")
print(cleaned)

## 7. Procesar todos los PDFs

Extraemos y limpiamos el texto de todos los PDFs encontrados en la carpeta configurada.

In [ ]:
def process_pdf(
    pdf_path: str,
    method: str = "pymupdf",
    **clean_kwargs
) -> list[dict]:
    """Extrae y limpia texto de un PDF completo."""
    extractor = EXTRACTORS.get(method)
    if extractor is None:
        raise ValueError(f"Método '{method}' no reconocido. Opciones: {list(EXTRACTORS.keys())}")

    # Extraer texto crudo
    raw_pages = extractor(pdf_path)

    # Limpiar cada página
    cleaned_pages = []
    for page_data in raw_pages:
        cleaned = clean_text(page_data["text"], **clean_kwargs)
        if cleaned:  # Solo incluir páginas con contenido
            cleaned_pages.append({
                "text": cleaned,
                "page": page_data["page"],
                "total_pages": page_data["total_pages"],
                "source": page_data["source"],
            })

    return cleaned_pages


# Procesar todos los PDFs
all_pages = []

for pdf_path in pdf_files:
    print(f"Procesando: {os.path.basename(pdf_path)}")
    try:
        pages = process_pdf(pdf_path, method="pymupdf")
        all_pages.extend(pages)
        print(f"  -> {len(pages)} páginas extraídas y limpiadas")
    except Exception as e:
        print(f"  -> Error: {e}")

print(f"\nTotal de páginas procesadas: {len(all_pages)}")

if all_pages:
    print(f"\nVista previa (primera página):")
    print(f"  Fuente: {all_pages[0]['source']}, Página: {all_pages[0]['page']}")
    print(f"  Texto: {all_pages[0]['text'][:300]}...")

## 8. Comparar métodos de extracción

Si quieres evaluar cuál método funciona mejor con tus PDFs, usa esta celda para comparar la salida de cada extractor.

In [ ]:
# Comparar los tres métodos de extracción en el primer PDF
if pdf_files:
    test_pdf = pdf_files[0]
    print(f"Comparando extractores en: {os.path.basename(test_pdf)}\n")

    for method_name, extractor_fn in EXTRACTORS.items():
        try:
            pages = extractor_fn(test_pdf)
            total_chars = sum(len(p["text"]) for p in pages)
            first_page_clean = clean_text(pages[0]["text"]) if pages else "(vacío)"

            print(f"--- {method_name.upper()} ---")
            print(f"  Páginas: {len(pages)}")
            print(f"  Caracteres totales: {total_chars:,}")
            print(f"  Primera página (limpia): {first_page_clean[:200]}...")
            print()
        except Exception as e:
            print(f"--- {method_name.upper()} ---")
            print(f"  Error: {e}\n")
else:
    print("No hay PDFs disponibles para comparar.")

## 9. Segmentación (Chunking) del texto

Los documentos largos deben dividirse en **chunks** (fragmentos) antes de generar embeddings. El tamaño del chunk afecta directamente la calidad del sistema RAG:

- **Chunks muy pequeños** → pierden contexto
- **Chunks muy grandes** → diluyen la información relevante

Usamos `RecursiveCharacterTextSplitter` de LangChain, que intenta dividir respetando la estructura del texto (párrafos > oraciones > palabras).

In [ ]:
# Configuración del text splitter
CHUNK_SIZE = 1000       # Tamaño máximo de cada chunk en caracteres
CHUNK_OVERLAP = 200     # Solapamiento entre chunks para mantener contexto

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", ", ", " ", ""],  # Prioridad de separación
)

print(f"Text Splitter configurado:")
print(f"  Tamaño de chunk: {CHUNK_SIZE} caracteres")
print(f"  Solapamiento: {CHUNK_OVERLAP} caracteres")

## 10. Crear objetos Document de LangChain

Convertimos las páginas limpias en objetos `Document` con metadata enriquecida, y luego los segmentamos. Estos objetos son directamente compatibles con el pipeline RAG del tutorial.

In [ ]:
def pages_to_documents(pages: list[dict]) -> list[Document]:
    """Convierte páginas limpias en objetos Document de LangChain."""
    documents = []
    for page in pages:
        doc = Document(
            page_content=page["text"],
            metadata={
                "source": page["source"],
                "page": page["page"],
                "total_pages": page["total_pages"],
            }
        )
        documents.append(doc)
    return documents


# Crear documentos a partir de las páginas limpias
page_documents = pages_to_documents(all_pages)
print(f"Documentos creados (por página): {len(page_documents)}")

# Segmentar en chunks
chunked_documents = text_splitter.split_documents(page_documents)
print(f"Documentos después de chunking: {len(chunked_documents)}")

if chunked_documents:
    print(f"\nEjemplo de chunk:")
    print(f"  Contenido: {chunked_documents[0].page_content[:200]}...")
    print(f"  Metadata: {chunked_documents[0].metadata}")
    
    # Estadísticas de chunks
    chunk_sizes = [len(doc.page_content) for doc in chunked_documents]
    print(f"\nEstadísticas de chunks:")
    print(f"  Mínimo: {min(chunk_sizes)} caracteres")
    print(f"  Máximo: {max(chunk_sizes)} caracteres")
    print(f"  Promedio: {sum(chunk_sizes) / len(chunk_sizes):.0f} caracteres")

## 11. Inspección de calidad

Revisa una muestra de los chunks generados para verificar la calidad de la limpieza.

In [ ]:
# Mostrar los primeros N chunks para inspección
N_PREVIEW = 5

for i, doc in enumerate(chunked_documents[:N_PREVIEW], 1):
    print(f"{'='*80}")
    print(f"CHUNK {i} | Fuente: {doc.metadata['source']} | Página: {doc.metadata['page']}")
    print(f"Longitud: {len(doc.page_content)} caracteres")
    print(f"{'='*80}")
    print(doc.page_content)
    print()

## 12. Exportar documentos limpios

### Opción A: Usar directamente en el notebook RAG

La variable `chunked_documents` contiene los objetos `Document` listos para usarse. En el notebook del Tutorial RAG, reemplaza la sección **"4. Prepare Sample Documents"** con:

```python
# En lugar de los documentos de ejemplo, usa los documentos limpios de PDFs:
documents = chunked_documents  # Generados por el notebook de limpieza
```

### Opción B: Guardar como archivo de texto para reutilizar

In [ ]:
# Guardar los textos limpios en archivos .txt (uno por PDF)
OUTPUT_FOLDER = "./pdfs_limpios"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if all_pages:
    # Agrupar páginas por fuente
    by_source = {}
    for page in all_pages:
        source = page["source"]
        if source not in by_source:
            by_source[source] = []
        by_source[source].append(page)

    for source, pages in by_source.items():
        txt_name = Path(source).stem + "_limpio.txt"
        txt_path = os.path.join(OUTPUT_FOLDER, txt_name)

        with open(txt_path, "w", encoding="utf-8") as f:
            for page in pages:
                f.write(f"--- Página {page['page']} ---\n")
                f.write(page["text"])
                f.write("\n\n")

        print(f"Guardado: {txt_path}")

    print(f"\nArchivos limpios guardados en '{OUTPUT_FOLDER}'")
else:
    print("No hay páginas procesadas para guardar.")

## 13. Función todo-en-uno para integración con RAG

Esta función encapsula todo el pipeline y devuelve una lista de `Document` lista para ser inyectada directamente en ChromaDB.

In [ ]:
def load_and_clean_pdfs(
    pdf_folder: str,
    extraction_method: str = "pymupdf",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    **clean_kwargs
) -> list[Document]:
    """
    Pipeline completo: busca PDFs -> extrae texto -> limpia -> segmenta -> Document[].

    Uso en el notebook RAG:
        documents = load_and_clean_pdfs("./pdfs")
        # Luego usar 'documents' donde el tutorial crea los documentos de ejemplo.
    """
    pdfs = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
    if not pdfs:
        print(f"No se encontraron PDFs en '{pdf_folder}'")
        return []

    all_pages = []
    for pdf_path in pdfs:
        try:
            pages = process_pdf(pdf_path, method=extraction_method, **clean_kwargs)
            all_pages.extend(pages)
            print(f"  {os.path.basename(pdf_path)}: {len(pages)} páginas")
        except Exception as e:
            print(f"  {os.path.basename(pdf_path)}: Error - {e}")

    # Convertir a Documents
    page_docs = pages_to_documents(all_pages)

    # Segmentar
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", ", ", " ", ""],
    )
    chunks = splitter.split_documents(page_docs)

    print(f"\nTotal: {len(all_pages)} páginas -> {len(chunks)} chunks")
    return chunks


# Ejemplo de uso
# documents = load_and_clean_pdfs("./pdfs", chunk_size=1000, chunk_overlap=200)
print("Función load_and_clean_pdfs() definida.")
print("\nPara usar en el Tutorial RAG:")
print('  documents = load_and_clean_pdfs("./pdfs")')

## Resumen

Este notebook proporciona un pipeline completo de limpieza de PDFs:

| Paso | Descripción |
|------|-------------|
| Extracción | PyMuPDF, PyPDF2 o pdfplumber |
| Limpieza de caracteres | Elimina basura Unicode, caracteres de control |
| Números de página | Elimina líneas con solo números |
| Encabezados/pies | Elimina patrones repetitivos |
| Hyphenation | Reconecta palabras cortadas por guión |
| Párrafos rotos | Une líneas que son continuación de párrafo |
| Espacios | Normaliza espacios múltiples y líneas vacías |
| Chunking | Segmenta en fragmentos óptimos para RAG |

